In [ ]:
import torch
import torchvision
import torch.utils.data
import torchvision.transforms.v2

transforms = torchvision.transforms.v2.Compose(
    [
        torchvision.transforms.v2.Resize(224),
        torchvision.transforms.v2.Grayscale(num_output_channels=3),
        torchvision.transforms.v2.ToImage(),
        torchvision.transforms.v2.ToDtype(torch.float32, scale=True),
        torchvision.transforms.v2.Normalize((0.1307,), (0.3081,)),
    ]
)
train_ds = torchvision.datasets.MNIST("mnist", train=True, download=True, transform=transforms)
test_ds = torchvision.datasets.MNIST("mnist", train=False, download=True, transform=transforms)

train_idxs = list(range(1000))
test_idxs = list(range(250))
train_y = train_ds.targets[train_idxs]
test_y = test_ds.targets[test_idxs]
train_ds = torch.utils.data.Subset(train_ds, indices=train_idxs)
test_ds = torch.utils.data.Subset(test_ds, indices=test_idxs)

In [ ]:
import zigzag.nn
import zigzag.utils
import zigzag.pipelines

PARAMS = [
    zigzag.pipelines.Params(k_neighbors=2, dimension=3),
    zigzag.pipelines.Params(k_neighbors=3, dimension=3),
    zigzag.pipelines.Params(k_neighbors=4, dimension=3),
    zigzag.pipelines.Params(k_neighbors=5, dimension=3),
    zigzag.pipelines.Params(k_neighbors=7, dimension=3),
    zigzag.pipelines.Params(k_neighbors=10, dimension=3),
    zigzag.pipelines.Params(k_neighbors=25, dimension=3)
]
dumper = zigzag.utils.UniversalDumper("zigzag_results/testing/vit_b_16/subset")
dumper.clear()

In [ ]:
pretrained_dumper = dumper.make_subdumper("pretrained")
model = torchvision.models.vit_b_16(num_classes=1000, weights=torchvision.models.ViT_B_16_Weights.DEFAULT)
model.heads = torch.nn.Identity()

zigzag.pipelines.validate_pretrained(model, train_ds, train_y, test_ds, test_y, pretrained_dumper)
hidden_states = pretrained_dumper.execute(zigzag.nn.collect_hidden_states, "hidden_states", model, train_ds)
zigzag.pipelines.analyze(hidden_states, PARAMS, pretrained_dumper)

In [ ]:
finetuned_dumper = dumper.make_subdumper("pretrained")

model = torchvision.models.vit_b_16(num_classes=1000, weights=torchvision.models.ViT_B_16_Weights.DEFAULT)
model.heads = pretrained_dumper.get_dump("trained_head")

zigzag.pipelines.train_validate(model, train_ds, test_ds, dumper)
hidden_states = dumper.execute(zigzag.nn.collect_hidden_states, "hidden_states", model, train_ds)
zigzag.pipelines.analyze(hidden_states, PARAMS, finetuned_dumper)